# Lesson 18 — Complex Queries (Capstone)

**Prerequisites:** Lessons 06 (subqueries & CTEs) and 07 (window functions). Lesson 13
(plan reading) helps for checking your work but isn't required.

You know each technique in isolation. This lesson is about **combining them**: taking a
multi-part business question and turning it into a correct, readable query — without
writing one giant unreadable statement.

## What you'll learn

- A repeatable strategy for decomposing a complex question into named steps
- Multi-CTE pipelines: layered aggregation, window functions over grouped results
- `CROSS APPLY` / `OUTER APPLY`: per-row subqueries and top-N-per-group
- `PIVOT` / `UNPIVOT`, and why conditional aggregation is often the better tool
- The classic patterns: top-N per group, de-duplication, gaps & islands, period comparisons

## Setup

Run `setup.sql` once. It creates the `lesson18` schema plus two tables AdventureWorks is
too clean to provide: `CustomerStaging` (contains deliberate duplicates) and `GymVisit`
(visit dates with gaps).

**Tooling note:** run the SQL cells with Azure Data Studio connected to the mssql-learn container (attach the SQL kernel to your saved localhost connection). This is a capstone — work through the concept cells, then run the worked-example cells **one section at a time**, reading each step's output before moving on, exactly as you would when building a complex query yourself.

In [ ]:
USE AdventureWorks2022;
GO

## The Case Study

Everything in this lesson builds one report:

> "For each sales territory: monthly revenue, month-over-month growth, the top 3 products,
> and each product's contribution % to territory revenue."

Don't try to write that as one statement. Decompose it:

```mermaid
flowchart TB
    A["OrderLines<br/><i>flatten header + detail to one row<br/>per line item, tag with month</i>"]
    B["MonthlyTerritory<br/><i>GROUP BY territory, month<br/>→ revenue per cell</i>"]
    C["WithGrowth<br/><i>LAG over the grouped rows<br/>→ previous month, growth %</i>"]
    D["Final SELECT<br/><i>join territory names,<br/>CROSS APPLY top 3 products,<br/>compute contribution %</i>"]
    A --> B --> C --> D
```

Each box becomes a named CTE. Each CTE is independently testable: while building, end the
query with `SELECT * FROM <that CTE>` and eyeball the output before adding the next layer.

## Concepts

### 1. Strategy: name the steps before writing SQL

Write the skeleton first — names and comments, no logic:

```sql
WITH OrderLines AS (
    -- one row per order line, tagged with territory + month
    SELECT 1 AS placeholder
),
MonthlyTerritory AS (
    -- revenue per territory per month
    SELECT 1 AS placeholder
),
WithGrowth AS (
    -- add previous month + growth % via LAG
    SELECT 1 AS placeholder
)
SELECT 1;  -- final shape
```

Then fill in one CTE at a time, testing each with `SELECT TOP (10) * FROM <cte>`.
A complex query you built this way is never scary — it's just steps.

### 2. Multi-CTE pipelines

Two rules of thumb:

- **Each CTE changes the grain once.** `OrderLines` is at line-item grain;
  `MonthlyTerritory` collapses it to territory×month. One grain change per step keeps
  every step explainable.
- **Window functions come *after* GROUP BY collapses the rows** — put the `GROUP BY` in
  one CTE and the `LAG`/`SUM() OVER` in the next. (An "aggregate of an aggregate" like
  *average monthly revenue* is just two pipeline steps: SUM per month, then AVG.)

### 3. CROSS APPLY / OUTER APPLY

`APPLY` runs the right side **once per row of the left side** — a subquery that can see
the current row. The killer use: top-N-per-group.

```sql
SELECT st.Name, big.SalesOrderID, big.TotalDue
FROM Sales.SalesTerritory AS st
CROSS APPLY (
    SELECT TOP (3) soh.SalesOrderID, soh.TotalDue
    FROM Sales.SalesOrderHeader AS soh
    WHERE soh.TerritoryID = st.TerritoryID   -- ← sees the outer row
    ORDER BY soh.TotalDue DESC
) AS big;
```

- `CROSS APPLY` = inner-join semantics: left rows with **no** right rows disappear.
- `OUTER APPLY` = left-join semantics: left rows survive with NULLs.

### 4. PIVOT, UNPIVOT, and conditional aggregation

`PIVOT` turns rows into columns, but demands an exact column list and exactly one
aggregate. **Conditional aggregation** does the same job with plain GROUP BY and mixes
freely with other aggregates:

```sql
SELECT st.Name,
       SUM(CASE WHEN YEAR(soh.OrderDate) = 2013 THEN soh.TotalDue END) AS [2013],
       SUM(CASE WHEN YEAR(soh.OrderDate) = 2014 THEN soh.TotalDue END) AS [2014],
       COUNT(*) AS TotalOrders            -- ← impossible inside a PIVOT
FROM Sales.SalesOrderHeader AS soh
JOIN Sales.SalesTerritory  AS st ON st.TerritoryID = soh.TerritoryID
GROUP BY st.Name;
```

Reach for `PIVOT` when the syntax reads cleaner; reach for conditional aggregation when
you need flexibility. `UNPIVOT` goes the other way (columns → rows) — handy for
normalizing wide tables.

### 5. The classic patterns

| Pattern | Tool | One-line recipe |
|---|---|---|
| Top-N per group | `ROW_NUMBER` or `CROSS APPLY` | Rank within partition, filter `rn <= N` — or APPLY a `TOP (N)` subquery |
| Ties at the boundary | `RANK` / `TOP WITH TIES` | `RANK` repeats numbers for ties; `WITH TIES` extends a single top-N |
| De-duplication | `ROW_NUMBER` + `DELETE` | Partition by the duplicate key, order by "which copy wins", delete `rn > 1` |
| Gaps & islands | `ROW_NUMBER` difference | `date - row_number` is constant within a consecutive run; GROUP BY that |
| Period comparison | `LAG` over grouped rows | GROUP to period grain first, then `LAG(...) OVER (PARTITION ... ORDER BY period)` |
| Share of total | windowed `SUM` | `value / SUM(value) OVER (PARTITION BY scope)` |

## Worked Examples

The cells below build the case study incrementally (examples 1–3), demo APPLY (4–5), PIVOT/UNPIVOT/conditional aggregation (6–8), the classic patterns (9–13), and assemble the full case-study query (14). Run them section by section, not all at once.

### Example 1 — Step A — flatten orders to the grain we need (one row per line item)

In [ ]:
WITH OrderLines AS (
    SELECT soh.TerritoryID,
           DATEFROMPARTS(YEAR(soh.OrderDate), MONTH(soh.OrderDate), 1) AS OrderMonth,
           sod.ProductID,
           sod.LineTotal
    FROM Sales.SalesOrderHeader AS soh
    JOIN Sales.SalesOrderDetail AS sod ON sod.SalesOrderID = soh.SalesOrderID
)
SELECT TOP (10) * FROM OrderLines;   -- debug technique: eyeball the step's shape/rows (use explicit columns in real code)

### Example 2 — Step B — first aggregation layer (territory × month)

In [ ]:
WITH OrderLines AS (
    SELECT soh.TerritoryID,
           DATEFROMPARTS(YEAR(soh.OrderDate), MONTH(soh.OrderDate), 1) AS OrderMonth,
           sod.ProductID,
           sod.LineTotal
    FROM Sales.SalesOrderHeader AS soh
    JOIN Sales.SalesOrderDetail AS sod ON sod.SalesOrderID = soh.SalesOrderID
),
MonthlyTerritory AS (
    SELECT TerritoryID, OrderMonth, SUM(LineTotal) AS MonthRevenue
    FROM OrderLines
    GROUP BY TerritoryID, OrderMonth
)
SELECT TOP (10) * FROM MonthlyTerritory ORDER BY TerritoryID, OrderMonth;

### Example 3 — Step C — window function OVER the grouped result (aggregate of an aggregate)

Note: LAG appears three times here for compactness. Example 14 shows the clean form: compute LAG once in its own CTE, then reference the column.

In [ ]:
WITH OrderLines AS (
    SELECT soh.TerritoryID,
           DATEFROMPARTS(YEAR(soh.OrderDate), MONTH(soh.OrderDate), 1) AS OrderMonth,
           sod.ProductID,
           sod.LineTotal
    FROM Sales.SalesOrderHeader AS soh
    JOIN Sales.SalesOrderDetail AS sod ON sod.SalesOrderID = soh.SalesOrderID
),
MonthlyTerritory AS (
    SELECT TerritoryID, OrderMonth, SUM(LineTotal) AS MonthRevenue
    FROM OrderLines
    GROUP BY TerritoryID, OrderMonth
)
SELECT TerritoryID, OrderMonth, MonthRevenue,
       LAG(MonthRevenue) OVER (PARTITION BY TerritoryID ORDER BY OrderMonth) AS PrevMonthRevenue,
       CAST(100.0 * (MonthRevenue - LAG(MonthRevenue) OVER (PARTITION BY TerritoryID ORDER BY OrderMonth))
            / NULLIF(LAG(MonthRevenue) OVER (PARTITION BY TerritoryID ORDER BY OrderMonth), 0)
            AS DECIMAL(10,1)) AS GrowthPct
FROM MonthlyTerritory
ORDER BY TerritoryID, OrderMonth;

### Example 4 — top-3-per-group with CROSS APPLY — each territory's 3 biggest orders

In [ ]:
SELECT st.Name AS Territory, big.SalesOrderID, big.TotalDue
FROM Sales.SalesTerritory AS st
CROSS APPLY (
    SELECT TOP (3) soh.SalesOrderID, soh.TotalDue
    FROM Sales.SalesOrderHeader AS soh
    WHERE soh.TerritoryID = st.TerritoryID
    ORDER BY soh.TotalDue DESC
) AS big
ORDER BY st.Name, big.TotalDue DESC;

### Example 5 — CROSS vs OUTER APPLY — CROSS silently drops rows with no match.

Products that have never sold disappear under CROSS APPLY; OUTER keeps them with NULLs.

In [ ]:
SELECT p.Name, lastSale.OrderDate
FROM Production.Product AS p
OUTER APPLY (
    SELECT TOP (1) soh.OrderDate
    FROM Sales.SalesOrderDetail AS sod
    JOIN Sales.SalesOrderHeader AS soh ON soh.SalesOrderID = sod.SalesOrderID
    WHERE sod.ProductID = p.ProductID
    ORDER BY soh.OrderDate DESC
) AS lastSale
WHERE lastSale.OrderDate IS NULL;   -- only possible with OUTER APPLY
-- (504 products in the catalog; 238 have never been sold)

### Example 6 — PIVOT — territory revenue by year, years as columns

In [ ]:
SELECT Name AS Territory, [2011], [2012], [2013], [2014]
FROM (
    SELECT st.Name, YEAR(soh.OrderDate) AS OrderYear, soh.TotalDue
    FROM Sales.SalesOrderHeader AS soh
    JOIN Sales.SalesTerritory  AS st ON st.TerritoryID = soh.TerritoryID
) AS src
PIVOT (SUM(TotalDue) FOR OrderYear IN ([2011], [2012], [2013], [2014])) AS pvt
ORDER BY Territory;

### Example 7 — Conditional aggregation instead of PIVOT

the same result with conditional aggregation — no PIVOT syntax, and you can mix in other aggregates (like the TotalOrders column) freely.

In [ ]:
SELECT st.Name AS Territory,
       SUM(CASE WHEN YEAR(soh.OrderDate) = 2011 THEN soh.TotalDue END) AS [2011],
       SUM(CASE WHEN YEAR(soh.OrderDate) = 2012 THEN soh.TotalDue END) AS [2012],
       SUM(CASE WHEN YEAR(soh.OrderDate) = 2013 THEN soh.TotalDue END) AS [2013],
       SUM(CASE WHEN YEAR(soh.OrderDate) = 2014 THEN soh.TotalDue END) AS [2014],
       COUNT(*) AS TotalOrders
FROM Sales.SalesOrderHeader AS soh
JOIN Sales.SalesTerritory  AS st ON st.TerritoryID = soh.TerritoryID
GROUP BY st.Name
ORDER BY Territory;

### Example 8 — UNPIVOT — wide salesperson measure columns back into rows

Note: UNPIVOT silently drops rows where the value is NULL (SalesQuota often is). If NULL rows matter, use conditional aggregation instead.

In [ ]:
SELECT BusinessEntityID, MeasureName, MeasureValue
FROM (
    SELECT BusinessEntityID,
           SalesQuota   AS CurrentQuota,
           SalesYTD     AS YearToDate,
           SalesLastYear AS LastYear
    FROM Sales.SalesPerson
) AS src
UNPIVOT (MeasureValue FOR MeasureName IN (CurrentQuota, YearToDate, LastYear)) AS unp
ORDER BY BusinessEntityID, MeasureName;

### Example 9 — top-N per group with ROW_NUMBER (compare with Example 4's APPLY version)

APPLY reads naturally for small fixed N; ROW_NUMBER lets you also keep the rank, filter ranges (rows 4-6), or switch to RANK/DENSE_RANK for ties.

In [ ]:
WITH Ranked AS (
    SELECT st.Name AS Territory, soh.SalesOrderID, soh.TotalDue,
           ROW_NUMBER() OVER (PARTITION BY st.TerritoryID ORDER BY soh.TotalDue DESC) AS rn
    FROM Sales.SalesOrderHeader AS soh
    JOIN Sales.SalesTerritory  AS st ON st.TerritoryID = soh.TerritoryID
)
SELECT Territory, SalesOrderID, TotalDue
FROM Ranked
WHERE rn <= 3
ORDER BY Territory, TotalDue DESC;

### Example 10 — ties — RANK keeps all rows that tie for a place; TOP WITH TIES

does the same for a single (non-grouped) top-N

In [ ]:
SELECT TOP (5) WITH TIES p.Name, p.ListPrice
FROM Production.Product AS p
ORDER BY p.ListPrice DESC;

### Example 11 — De-duplication with ROW_NUMBER (preview)

de-duplication with ROW_NUMBER (preview only — the DELETE version is exercise 6). rn = 1 marks the keeper per email; rows with rn > 1 are the copies a DELETE would remove.

In [ ]:
WITH Ranked AS (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY Email
                                 ORDER BY LoadDate DESC, StagingID DESC) AS rn
    FROM lesson18.CustomerStaging
)
SELECT StagingID, Email, LoadDate, rn
FROM Ranked
WHERE rn > 1
ORDER BY Email, rn;

### Example 12 — gaps & islands — consecutive visit streaks per member.

Trick: VisitDate minus its row number is CONSTANT within a consecutive run.

In [ ]:
WITH Numbered AS (
    SELECT MemberID, VisitDate,
           DATEADD(DAY,
                   -ROW_NUMBER() OVER (PARTITION BY MemberID ORDER BY VisitDate),
                   VisitDate) AS StreakKey
    FROM lesson18.GymVisit
)
SELECT MemberID,
       MIN(VisitDate) AS StreakStart,
       MAX(VisitDate) AS StreakEnd,
       COUNT(*)       AS StreakDays
FROM Numbered
GROUP BY MemberID, StreakKey
ORDER BY MemberID, StreakStart;

### Example 13 — running total + share of period (windowed SUM at two scopes)

In [ ]:
WITH MonthlySales AS (
    SELECT DATEFROMPARTS(YEAR(OrderDate), MONTH(OrderDate), 1) AS OrderMonth,
           SUM(TotalDue) AS Revenue
    FROM Sales.SalesOrderHeader
    GROUP BY DATEFROMPARTS(YEAR(OrderDate), MONTH(OrderDate), 1)
)
SELECT OrderMonth, Revenue,
       SUM(Revenue) OVER (ORDER BY OrderMonth
                          ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS RunningTotal,
       CAST(100.0 * Revenue / SUM(Revenue) OVER () AS DECIMAL(5,2)) AS PctOfAllTime
FROM MonthlySales
ORDER BY OrderMonth;

### The assembled case study

Pipeline + window functions + APPLY together.

### Example 14 — the full answer to the business question

In [ ]:
WITH OrderLines AS (
    SELECT soh.TerritoryID,
           DATEFROMPARTS(YEAR(soh.OrderDate), MONTH(soh.OrderDate), 1) AS OrderMonth,
           sod.ProductID,
           sod.LineTotal
    FROM Sales.SalesOrderHeader AS soh
    JOIN Sales.SalesOrderDetail AS sod ON sod.SalesOrderID = soh.SalesOrderID
),
MonthlyTerritory AS (
    SELECT TerritoryID, OrderMonth, SUM(LineTotal) AS MonthRevenue
    FROM OrderLines
    GROUP BY TerritoryID, OrderMonth
),
WithGrowth AS (
    SELECT TerritoryID, OrderMonth, MonthRevenue,
           LAG(MonthRevenue) OVER (PARTITION BY TerritoryID ORDER BY OrderMonth) AS PrevMonthRevenue
    FROM MonthlyTerritory
)
SELECT st.Name AS Territory,
       g.OrderMonth,
       g.MonthRevenue,
       CAST(100.0 * (g.MonthRevenue - g.PrevMonthRevenue)
            / NULLIF(g.PrevMonthRevenue, 0) AS DECIMAL(10,1)) AS GrowthPct,
       top3.ProductName,
       top3.ProductRevenue,
       CAST(100.0 * top3.ProductRevenue / g.MonthRevenue AS DECIMAL(5,1)) AS ContributionPct
FROM WithGrowth AS g
JOIN Sales.SalesTerritory AS st ON st.TerritoryID = g.TerritoryID
CROSS APPLY (
    SELECT TOP (3) p.Name AS ProductName, SUM(ol.LineTotal) AS ProductRevenue
    FROM OrderLines AS ol
    JOIN Production.Product AS p ON p.ProductID = ol.ProductID
    WHERE ol.TerritoryID = g.TerritoryID
      AND ol.OrderMonth  = g.OrderMonth
    GROUP BY p.Name
    ORDER BY SUM(ol.LineTotal) DESC
) AS top3
ORDER BY Territory, g.OrderMonth, top3.ProductRevenue DESC;

## Pitfalls

- **CTEs are not temp tables.** Referencing the same CTE twice executes it twice. If a
  step is expensive and reused, materialize it into a `#temp` table instead.
- **CROSS APPLY silently drops rows** when the inner query returns nothing. If the left
  side must survive, you wanted `OUTER APPLY`.
- **`ORDER BY` inside a CTE or subquery does not order the result** (it's only legal
  with `TOP`). Order in the outermost SELECT.
- **Window functions can't go in `WHERE`** — they're computed at SELECT time (see
  cheatsheet 00's logical order). Compute in a CTE, filter in the next step.
- **PIVOT needs literal column values.** Dynamic categories require dynamic SQL — usually
  a sign you want conditional aggregation or to pivot in the application layer.

## Cheatsheet link

See `cheatsheets/01-tsql-syntax.md`, `cheatsheets/04-window-functions.md`, and the
logical-execution-order diagram in `cheatsheets/00-how-mssql-works.md`.

## Exercises

Open `exercises.ipynb` — eight exercises ramping from CTE-pipeline warm-ups to a final boss that mirrors the case study on Purchasing data. Solutions with explanations are in `exercises-solutions.ipynb`.